In [1]:
import pandas as pd
import duckdb
from IPython.display import display

In [2]:
path = (r"\Users\DELL\Downloads\archive")
aisles = pd.read_csv(r"\Users\DELL\Downloads\archive\aisles.csv")
departments = pd.read_csv(r"\Users\DELL\Downloads\archive\departments.csv")
order_products = pd.read_csv(r"\Users\DELL\Downloads\archive\order_products__prior.csv")
orders = pd.read_csv(r"\Users\DELL\Downloads\archive\orders.csv")
products = pd.read_csv(r"\Users\DELL\Downloads\archive\products.csv")
departments

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol
5,6,international
6,7,beverages
7,8,pets
8,9,dry goods pasta
9,10,bulk


1. Create a list of product names alongside their respective department names.

In [3]:
conn = duckdb.connect()
conn.sql(
    """
    SELECT p.product_name,d.department
    from products p
    LEFT JOIN departments d
    ON p.department_id = d.department_id
    """)

┌───────────────────────────────────────────────────────────────────┬─────────────────┐
│                           product_name                            │   department    │
│                              varchar                              │     varchar     │
├───────────────────────────────────────────────────────────────────┼─────────────────┤
│ Chocolate Sandwich Cookies                                        │ snacks          │
│ All-Seasons Salt                                                  │ pantry          │
│ Robust Golden Unsweetened Oolong Tea                              │ beverages       │
│ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce │ frozen          │
│ Green Chile Anytime Sauce                                         │ pantry          │
│ Dry Nose Oil                                                      │ personal care   │
│ Pure Coconut Water With Orange                                    │ beverages       │
│ Cut Russet Potatoes Steam N' M

2. Retrieve a list showing order_id and the names of the products contained in each order

In [4]:
order_products

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0
...,...,...,...,...
32434484,3421083,39678,6,1
32434485,3421083,11352,7,0
32434486,3421083,4600,8,0
32434487,3421083,24852,9,1


In [5]:
conn.sql(
    """
    SELECT o.order_id, p.product_name
    from order_products o
    LEFT JOIN products p
    ON o.product_id = p.product_id
   
    """)

┌──────────┬───────────────────────────────────────────────────────────────────────────────────┐
│ order_id │                                   product_name                                    │
│  int64   │                                      varchar                                      │
├──────────┼───────────────────────────────────────────────────────────────────────────────────┤
│    10826 │ Apple Lemon Cayenne Ginger                                                        │
│    10826 │ Organic Turmeric Tonic Juice                                                      │
│    10826 │ Arugula, Kale, Apple, Romaine, Celery, Cucumber, Lemon, Ginger Cold Pressed Juice │
│    10826 │ Carrot Golden Beet Juice                                                          │
│    10826 │ Organic Grade A Free Range Large Brown Eggs                                       │
│    10826 │ Organic Chicken Noodle Soup                                                       │
│    10826 │ All Natural Cinna

3. List all products and the name of the aisle they belong to.

In [6]:
aisles

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation
...,...,...
129,130,hot cereal pancake mixes
130,131,dry pasta
131,132,beauty
132,133,muscles joints pain relief


In [7]:
conn.sql("""
    SELECT 
    p.product_name,
    a.aisle
    FROM products p
    LEFT JOIN aisles a
    ON p.aisle_id = a.aisle_id;
    """)

┌───────────────────────────────────────────────────────────────────┬───────────────────────────────┐
│                           product_name                            │             aisle             │
│                              varchar                              │            varchar            │
├───────────────────────────────────────────────────────────────────┼───────────────────────────────┤
│ Chocolate Sandwich Cookies                                        │ cookies cakes                 │
│ All-Seasons Salt                                                  │ spices seasonings             │
│ Robust Golden Unsweetened Oolong Tea                              │ tea                           │
│ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce │ frozen meals                  │
│ Green Chile Anytime Sauce                                         │ marinades meat preparation    │
│ Dry Nose Oil                                                      │ cold flu all

4. Calculate the total number of items sold per department name.

In [11]:
conn.sql("""
    SELECT 
    d.department,
    COUNT(*) AS total_items_sold
FROM order_products op
JOIN products p
    ON op.product_id = p.product_id
JOIN departments d
    ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY total_items_sold DESC;
""")

┌─────────────────┬──────────────────┐
│   department    │ total_items_sold │
│     varchar     │      int64       │
├─────────────────┼──────────────────┤
│ produce         │          9479291 │
│ dairy eggs      │          5414016 │
│ snacks          │          2887550 │
│ beverages       │          2690129 │
│ frozen          │          2236432 │
│ pantry          │          1875577 │
│ bakery          │          1176787 │
│ canned goods    │          1068058 │
│ deli            │          1051249 │
│ dry goods pasta │           866627 │
│ household       │           738666 │
│ breakfast       │           709569 │
│ meat seafood    │           708931 │
│ personal care   │           447123 │
│ babies          │           423802 │
│ international   │           269253 │
│ alcohol         │           153696 │
│ pets            │            97724 │
│ missing         │            69145 │
│ other           │            36291 │
│ bulk            │            34573 │
└─────────────────┴──────

5. Display user_id, order_id, and product_name for the first 20 records in the dataset.

In [14]:
conn.sql(
    """
    SELECT o.user_id,op.order_id,p.product_name
    FROM order_products op
    JOIN orders o
    ON op.order_id = o.order_id
    JOIN products p
    ON op.product_id = p.product_id
    LIMIT 20;
    """
)

┌─────────┬──────────┬───────────────────────────────────────────────────────┐
│ user_id │ order_id │                     product_name                      │
│  int64  │  int64   │                        varchar                        │
├─────────┼──────────┼───────────────────────────────────────────────────────┤
│  202279 │        2 │ Organic Egg Whites                                    │
│  202279 │        2 │ Michigan Organic Kale                                 │
│  202279 │        2 │ Garlic Powder                                         │
│  202279 │        2 │ Coconut Butter                                        │
│  202279 │        2 │ Natural Sweetener                                     │
│  202279 │        2 │ Carrots                                               │
│  202279 │        2 │ Original Unflavored Gelatine Mix                      │
│  202279 │        2 │ All Natural No Stir Creamy Almond Butter              │
│  202279 │        2 │ Classic Blend Cole Slaw      